# FMN Project 1 — LightGBM Forecasting

## Objective
Test whether a pooled LightGBM model materially improves lead time demand forecasting over the simple baselines already established.

The model is evaluated on the business horizon used by the inventory decision engine: **expected demand during each SKU's lead time**.

### Guardrails
* No random train/test split.
* Features use information available at the forecast origin only.
* Future demand is used only to construct evaluation targets.
* The six unresolved demand values are not artificially imputed into model truth.
* LightGBM must beat the simple baselines sufficiently to justify added complexity.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_daily_data.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values(["sku_id", "date"]).reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"SKUs: {df.sku_id.nunique()}")
print(f"Date range: {df.date.min().date()} to {df.date.max().date()}")

Rows: 4,536
SKUs: 28
Date range: 2026-01-01 to 2026-06-29


### Finding
The forecasting dataset contains 28 SKUs across the six month assessment period. It is the validated prepared dataset rather than the raw CSV.

**Modelling implication:** forecasting should use the prepared demand series and preserve provenance of reconstructed versus original demand.

In [2]:
unresolved = df[df["units_sold"].isna()]
print("Unresolved demand values:", len(unresolved))
print(unresolved[["date","sku_id","closing_stock","units_received","units_sold_source"]].to_string(index=False))

Unresolved demand values: 6
      date   sku_id  closing_stock  units_received units_sold_source
2026-01-01 SKU-1001         3108.0               0          observed
2026-05-15 SKU-1002            0.0               0          observed
2026-05-29 SKU-1009         8468.0            5799          observed
2026-06-03 SKU-1017            0.0               0          observed
2026-01-29 SKU-1022            0.0               0          observed
2026-01-30 SKU-1022            0.0               0          observed


### Finding
Six `units_sold` observations remain unresolved after preparation. These cannot be safely inferred uniquely, particularly where stock is clipped at zero.

**Decision:** do not fabricate these values. They are excluded when constructing supervised targets that require complete future demand.

## 1. Forecasting formulation

The LightGBM model predicts a **demand ratio**:

`next lead time demand / (28 day trailing average × lead time)`

Predicted lead time demand is:

`predicted ratio × trailing 28 day average × lead time`

This allows one pooled model to learn demand dynamics while the trailing 28 day level retains SKU scale.

In [3]:
work = df.copy()
work["day_idx"] = work.groupby("sku_id").cumcount() + 1
established = work[work["sku_status"].eq("established")].copy()

FEATURES = [
    "lag_1", "lag_7", "lag_14",
    "roll_mean_7", "roll_mean_14", "roll_mean_28",
    "roll_std_7", "roll_std_14", "roll_std_28",
    "trend_ratio_14_28", "day_of_week", "lead_time_days",
    "category_code",
]

def build_supervised_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Build leakage controlled pooled training rows for lead time demand forecasting."""
    rows = []
    for sku_id, group in data.groupby("sku_id", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        valid_lt = group["lead_time_days"].dropna()
        if valid_lt.empty:
            continue
        lead_time = int(round(valid_lt.mode().iloc[0]))

        for origin_idx in range(28, len(group) - lead_time):
            history = group.loc[:origin_idx, "units_sold"].astype(float)
            future = group.loc[origin_idx + 1: origin_idx + lead_time, "units_sold"].astype(float)

            if history.tail(28).isna().any() or future.isna().any():
                continue

            base = history.tail(28).mean()
            if pd.isna(base) or base <= 0:
                continue

            rows.append({
                "sku_id": sku_id,
                "origin_date": group.loc[origin_idx, "date"],
                "category": group.loc[origin_idx, "category"],
                "lag_1": history.iloc[-1],
                "lag_7": history.iloc[-7],
                "lag_14": history.iloc[-14],
                "roll_mean_7": history.tail(7).mean(),
                "roll_mean_14": history.tail(14).mean(),
                "roll_mean_28": base,
                "roll_std_7": history.tail(7).std(),
                "roll_std_14": history.tail(14).std(),
                "roll_std_28": history.tail(28).std(),
                "trend_ratio_14_28": history.tail(14).mean() / base,
                "day_of_week": group.loc[origin_idx, "date"].dayofweek,
                "lead_time_days": lead_time,
                "target_ratio": future.sum() / (base * lead_time),
                "actual_lead_demand": future.sum(),
                "baseline_28": base * lead_time,
                "lead_time": lead_time,
            })

    result = pd.DataFrame(rows)
    result["category_code"] = result["category"].astype("category").cat.codes
    return result

supervised = build_supervised_frame(established)
supervised["calendar_day"] = (
    supervised["origin_date"] - pd.Timestamp("2026-01-01")
).dt.days + 1

print("Supervised rows:", len(supervised))
print("Lead time groups:", supervised.lead_time.value_counts().sort_index().to_dict())


Supervised rows: 3458
Lead time groups: {3: 509, 5: 294, 7: 1015, 10: 568, 14: 1072}


### Finding
Only origins with 28 days of history and complete future lead time demand enter the supervised set. This reduces sample size deliberately to protect against leakage and fabricated targets.

**Modelling implication:** the model learns a general demand adjustment ratio rather than memorising raw SKU demand levels.

## 2. Time based tuning and test design

* Tuning window: calendar days 90–119
* Final test: calendar day 120 onward
* At each origin, only earlier origins are available for training.
* This is an expanding rolling origin evaluation, not a random split.

In [4]:
def wape(actual, predicted):
    """Return weighted absolute percentage error."""
    denominator = np.abs(actual).sum()
    return np.abs(actual - predicted).sum() / denominator if denominator else np.nan

def fit_predict_expanding(origins, params):
    """Fit an expanding pooled LightGBM model at each forecast origin."""
    predictions = []
    for origin in sorted(origins):
        train = supervised[supervised["origin_date"] < origin]
        test = supervised[supervised["origin_date"] == origin].copy()
        if train.empty or test.empty:
            continue

        model = LGBMRegressor(
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )
        model.fit(train[FEATURES], train["target_ratio"])
        test["pred_ratio"] = np.clip(model.predict(test[FEATURES]), 0, None)
        test["pred_demand"] = test["pred_ratio"] * test["baseline_28"]
        predictions.append(test)

    return pd.concat(predictions, ignore_index=True)

tuning_origins = sorted(
    supervised.loc[supervised.calendar_day.between(90, 119), "origin_date"].unique()
)
test_origins = sorted(
    supervised.loc[supervised.calendar_day >= 120, "origin_date"].unique()
)

print("Tuning origins:", len(tuning_origins))
print("Test origins:", len(test_origins))


Tuning origins: 30
Test origins: 58


In [5]:
parameter_grid = [
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 7, "min_child_samples": 20},
    {"n_estimators": 200, "learning_rate": 0.03, "num_leaves": 7, "min_child_samples": 20},
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 15, "min_child_samples": 20},
    {"n_estimators": 200, "learning_rate": 0.03, "num_leaves": 15, "min_child_samples": 20},
]

tuning_results = []
for params in parameter_grid:
    pred = fit_predict_expanding(tuning_origins, params)
    tuning_results.append({
        **params,
        "wape": wape(pred["actual_lead_demand"], pred["pred_demand"]),
        "rows": len(pred),
    })

tuning_results = pd.DataFrame(tuning_results).sort_values("wape")
tuning_results


,n_estimators,learning_rate,num_leaves,min_child_samples,wape,rows
3,200,0.03,15,20,0.062336,750
2,100,0.05,15,20,0.063055,750
1,200,0.03,7,20,0.065874,750
0,100,0.05,7,20,0.066418,750


### Finding
Among the tested configurations, the strongest tuning result uses 200 estimators, learning rate 0.03, 15 leaves and minimum child samples of 20.

The tuning window is used only for configuration. The day 120 onward period remains the final test.

In [6]:
best_params = tuning_results.iloc[0][
    ["n_estimators", "learning_rate", "num_leaves", "min_child_samples"]
].to_dict()
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["num_leaves"] = int(best_params["num_leaves"])
best_params["min_child_samples"] = int(best_params["min_child_samples"])

lgbm_test = fit_predict_expanding(test_origins, best_params)

print("Test rows:", len(lgbm_test))
print("LightGBM lead time WAPE:", f"{wape(lgbm_test.actual_lead_demand, lgbm_test.pred_demand):.2%}")


Test rows: 1212
LightGBM lead time WAPE: 6.52%


## 3. Apples to apples baseline comparison

All baselines below are evaluated on the **same SKU origins** as LightGBM.

The weekday adjusted baseline forecasts each future day using the 28 day demand level multiplied by the historical weekday seasonal index.

In [7]:
baseline_rows = []

for _, row in lgbm_test.iterrows():
    group = established[established.sku_id.eq(row.sku_id)].sort_values("date").reset_index(drop=True)
    origin_idx = group.index[group.date.eq(row.origin_date)][0]
    history = group.loc[:origin_idx, "units_sold"].astype(float)

    ma14 = history.tail(14).mean() * row.lead_time
    ma28 = history.tail(28).mean() * row.lead_time

    last28 = group.loc[max(0, origin_idx - 27):origin_idx].copy()
    overall = last28.units_sold.mean()
    seasonal = last28.assign(dow=last28.date.dt.dayofweek).groupby("dow").units_sold.mean() / overall

    future_dates = pd.date_range(
        row.origin_date + pd.Timedelta(days=1),
        periods=int(row.lead_time),
    )
    weekday_adjusted = sum(
        overall * seasonal.get(day.dayofweek, 1.0)
        for day in future_dates
    )

    baseline_rows.append({
        "sku_id": row.sku_id,
        "origin_date": row.origin_date,
        "actual": row.actual_lead_demand,
        "ma14": ma14,
        "ma28": ma28,
        "weekday28": weekday_adjusted,
    })

baseline_test = pd.DataFrame(baseline_rows)

comparison = pd.DataFrame({
    "Model": [
        "14 day MA",
        "28 day MA",
        "Weekday adjusted 28 day MA",
        "Pooled LightGBM",
    ],
    "Lead time WAPE": [
        wape(baseline_test.actual, baseline_test.ma14),
        wape(baseline_test.actual, baseline_test.ma28),
        wape(baseline_test.actual, baseline_test.weekday28),
        wape(lgbm_test.actual_lead_demand, lgbm_test.pred_demand),
    ],
})
comparison["Lead time WAPE"] = comparison["Lead time WAPE"] * 100
comparison


,Model,Lead time WAPE
0,14 day MA,8.609080
1,28 day MA,7.336329
2,Weekday adjusted 28 day MA,7.284924
3,Pooled LightGBM,6.519762


### Finding

On the untouched test period:

* 14 day MA: **8.61%**
* 28 day MA: **7.34%**
* Weekday adjusted 28 day MA: **7.28%**
* Pooled LightGBM: **6.52%**

LightGBM improves on the strongest simple baseline by about **0.77 percentage points**, or **10.6% relative WAPE reduction**.

**Decision:** use pooled LightGBM as the forecasting champion for this prototype. Retain the weekday adjusted 28 day MA as the transparent fallback benchmark.

In [8]:
merged = lgbm_test[
    ["sku_id", "origin_date", "lead_time", "actual_lead_demand", "pred_demand"]
].merge(
    baseline_test,
    on=["sku_id", "origin_date"],
    how="left",
)

lead_metrics = []
for lead_time, group in merged.groupby("lead_time"):
    lead_metrics.append({
        "Lead time": int(lead_time),
        "Rows": len(group),
        "14 day MA WAPE": wape(group.actual_lead_demand, group.ma14),
        "28 day MA WAPE": wape(group.actual_lead_demand, group.ma28),
        "Weekday 28 day WAPE": wape(group.actual_lead_demand, group.weekday28),
        "LightGBM WAPE": wape(group.actual_lead_demand, group.pred_demand),
        "LightGBM bias": (
            group.pred_demand.sum() - group.actual_lead_demand.sum()
        ) / group.actual_lead_demand.sum(),
    })

lead_metrics = pd.DataFrame(lead_metrics).sort_values("Lead time")
lead_metrics


,Lead time,Rows,14 day MA WAPE,28 day MA WAPE,Weekday 28 day WAPE,LightGBM WAPE,LightGBM bias
0,3,174,0.112710,0.104663,0.098739,0.092544,0.005430
1,5,112,0.068383,0.063459,0.068551,0.068612,0.014437
2,7,378,0.103112,0.095847,0.095847,0.083530,0.007666
3,10,204,0.077892,0.064567,0.062043,0.059945,0.015685
4,14,344,0.083559,0.068369,0.068369,0.059329,-0.015246


### Finding

The 3 day lead time group remains the most difficult group for LightGBM, while performance is stronger for the 10 and 14 day groups.

**Business implication:** the product should expose forecast evidence and uncertainty rather than imply that model accuracy is uniform across all SKUs.

In [9]:
bias = (
    lgbm_test["pred_demand"].sum() - lgbm_test["actual_lead_demand"].sum()
) / lgbm_test["actual_lead_demand"].sum()

print(f"Overall LightGBM bias: {bias:.2%}")

final_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    **best_params,
)
final_model.fit(supervised[FEATURES], supervised["target_ratio"])

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance


Overall LightGBM bias: -0.27%


,feature,importance
5,roll_mean_28,468
8,roll_std_28,406
7,roll_std_14,386
11,lead_time_days,256
9,trend_ratio_14_28,242
4,roll_mean_14,213
6,roll_std_7,210
3,roll_mean_7,207
10,day_of_week,154
12,category_code,86


### Finding

Feature importance gives a useful diagnostic of which recent demand signals the pooled model relies on. It is **not** treated as the planner facing explanation.

The product will explain inventory risk using deterministic drivers such as projected stock, demand, coverage, lead time and replenishment evidence.

## 4. Final model decision

### Champion
**Pooled LightGBM**

* Lead time WAPE: **6.52%**
* Strongest simple baseline: **7.28%**
* Relative WAPE reduction: **~10.6%**

### Fallback
**Weekday adjusted 28 day moving average**

This remains valuable because it is transparent, lightweight and close to the champion.

### Limitation
The assessment dataset covers only six months and behaves like a controlled prototype dataset. These results support the assessment prototype, not a claim of production performance on live FMN data.

## 5. Modelling implications for the Risk Engine

The forecast is not the product by itself. It becomes useful when combined with replenishment evidence:

```text
Forecast
   ↓
Lead time demand
   ↓
Expected replenishment
   ↓
Projected stock
   ↓
Risk state
   ↓
Attention ranking
   ↓
Structured drivers
```

### Next stage

Build the replenishment and stock projection layer, then backtest the risk states and attention ranking against historical stockout and excess inventory outcomes.